In [6]:
# Import Pandas for data validation and cleaning
import pandas as pd
import numpy as np

# Load both CSV files because this is a new notebook
trips = pd.read_csv(r"C:\Users\M.SANTHOSH\Downloads\Uber Trip Details.csv")
locations = pd.read_csv(r"C:\Users\M.SANTHOSH\Downloads\Location Table.csv")

# Create copies so the original datasets remain unchanged
trips_clean = trips.copy()
locations_clean = locations.copy()

# Confirm that both datasets loaded successfully
print("Trip dataset:", trips_clean.shape)
print("Location dataset:", locations_clean.shape)

Trip dataset: (103728, 11)
Location dataset: (265, 3)


In [7]:
# Inspect column names to identify the fields used in validation
print("Trip columns:")
print(trips_clean.columns.tolist())

print("\nLocation columns:")
print(locations_clean.columns.tolist())

Trip columns:
['Trip ID', 'Pickup Time', 'Drop Off Time', 'passenger_count', 'trip_distance', 'PULocationID', 'DOLocationID', 'fare_amount', 'Surge Fee', 'Vehicle', 'Payment_type']

Location columns:
['LocationID', 'Location', 'City']


In [8]:
# Count missing values in every trip column to identify incomplete records
print("Missing values in trip dataset:")
print(trips_clean.isnull().sum())

# Count missing values in every location column
print("\nMissing values in location dataset:")
print(locations_clean.isnull().sum())

# Calculate the percentage of missing values to understand their impact
print("\nMissing value percentage in trip dataset:")
print((trips_clean.isnull().mean() * 100).round(2))

Missing values in trip dataset:
Trip ID            0
Pickup Time        0
Drop Off Time      0
passenger_count    0
trip_distance      0
PULocationID       0
DOLocationID       0
fare_amount        0
Surge Fee          0
Vehicle            0
Payment_type       0
dtype: int64

Missing values in location dataset:
LocationID    0
Location      1
City          2
dtype: int64

Missing value percentage in trip dataset:
Trip ID            0.0
Pickup Time        0.0
Drop Off Time      0.0
passenger_count    0.0
trip_distance      0.0
PULocationID       0.0
DOLocationID       0.0
fare_amount        0.0
Surge Fee          0.0
Vehicle            0.0
Payment_type       0.0
dtype: float64


In [9]:
# Count fully duplicated trip rows to identify repeated records
print("Duplicate trip rows:", trips_clean.duplicated().sum())

# Count fully duplicated location rows to identify repeated location records
print("Duplicate location rows:", locations_clean.duplicated().sum())

# Check whether Trip ID values are duplicated
print("Duplicate Trip IDs:", trips_clean["Trip ID"].duplicated().sum())

# Check whether LocationID values are duplicated
print("Duplicate Location IDs:", locations_clean["LocationID"].duplicated().sum())

Duplicate trip rows: 0
Duplicate location rows: 0
Duplicate Trip IDs: 0
Duplicate Location IDs: 0


In [11]:
# Display sample pickup times to identify the date format
print(trips_clean["Pickup Time"].head(10))

# Display sample drop-off times to identify the date format
print(trips_clean["Drop Off Time"].head(10))

0   NaT
1   NaT
2   NaT
3   NaT
4   NaT
5   NaT
6   NaT
7   NaT
8   NaT
9   NaT
Name: Pickup Time, dtype: datetime64[s]
0   NaT
1   NaT
2   NaT
3   NaT
4   NaT
5   NaT
6   NaT
7   NaT
8   NaT
9   NaT
Name: Drop Off Time, dtype: datetime64[s]


In [12]:
# Display current data types to identify columns that need conversion
print(trips_clean.dtypes)

# Convert pickup and drop-off columns to datetime for time-based analysis
trips_clean["Pickup Time"] = pd.to_datetime(
    trips_clean["Pickup Time"], errors="coerce"
)

trips_clean["Drop Off Time"] = pd.to_datetime(
    trips_clean["Drop Off Time"], errors="coerce"
)

# Convert numeric trip fields to numeric types for calculations
numeric_columns = [
    "passenger_count",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "fare_amount",
    "Surge Fee"
]

for column in numeric_columns:
    trips_clean[column] = pd.to_numeric(
        trips_clean[column], errors="coerce"
    )

# Verify the updated data types
print(trips_clean.dtypes)

Trip ID                    int64
Pickup Time        datetime64[s]
Drop Off Time      datetime64[s]
passenger_count            int64
trip_distance            float64
PULocationID               int64
DOLocationID               int64
fare_amount              float64
Surge Fee                float64
Vehicle                      str
Payment_type                 str
dtype: object
Trip ID                    int64
Pickup Time        datetime64[s]
Drop Off Time      datetime64[s]
passenger_count            int64
trip_distance            float64
PULocationID               int64
DOLocationID               int64
fare_amount              float64
Surge Fee                float64
Vehicle                      str
Payment_type                 str
dtype: object


In [13]:
# Convert pickup time using the known date format
trips_clean["Pickup Time"] = pd.to_datetime(
    trips_clean["Pickup Time"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

# Convert drop-off time using the known date format
trips_clean["Drop Off Time"] = pd.to_datetime(
    trips_clean["Drop Off Time"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)

# Check whether any values failed to convert
print("Invalid pickup times:", trips_clean["Pickup Time"].isna().sum())
print("Invalid drop-off times:", trips_clean["Drop Off Time"].isna().sum())

Invalid pickup times: 103728
Invalid drop-off times: 103728


In [14]:
# Count records with zero or negative trip distances
print(
    "Invalid trip distances:",
    (trips_clean["trip_distance"] <= 0).sum()
)

# Count records with negative fares
print(
    "Negative fares:",
    (trips_clean["fare_amount"] < 0).sum()
)

# Count records with negative surge fees
print(
    "Negative surge fees:",
    (trips_clean["Surge Fee"] < 0).sum()
)

# Count records where drop-off occurs before pickup
print(
    "Invalid trip times:",
    (
        trips_clean["Drop Off Time"] < trips_clean["Pickup Time"]
    ).sum()
)

# Check passenger counts that are zero or negative
print(
    "Invalid passenger counts:",
    (trips_clean["passenger_count"] <= 0).sum()
)

Invalid trip distances: 0
Negative fares: 0
Negative surge fees: 0
Invalid trip times: 0
Invalid passenger counts: 0


In [15]:
# Collect valid location IDs from the location dataset
valid_location_ids = set(locations_clean["LocationID"].dropna())

# Find pickup IDs that do not exist in the location dataset
invalid_pickups = ~trips_clean["PULocationID"].isin(valid_location_ids)

# Find drop-off IDs that do not exist in the location dataset
invalid_dropoffs = ~trips_clean["DOLocationID"].isin(valid_location_ids)

# Report unmatched location IDs without deleting any trip records
print("Trips with unmatched pickup IDs:", invalid_pickups.sum())
print("Trips with unmatched drop-off IDs:", invalid_dropoffs.sum())

Trips with unmatched pickup IDs: 0
Trips with unmatched drop-off IDs: 0


In [16]:
# Summarize the dataset after type conversion and validation checks
print("Original trip rows:", len(trips))
print("Rows in working copy:", len(trips_clean))
print("Remaining missing values:", trips_clean.isnull().sum().sum())
print("Remaining duplicate rows:", trips_clean.duplicated().sum())

# Display sample records to verify that the data is still readable
display(trips_clean.head())

Original trip rows: 103728
Rows in working copy: 103728
Remaining missing values: 207456
Remaining duplicate rows: 0


,Trip ID,Pickup Time,Drop Off Time,passenger_count,trip_distance,PULocationID,DOLocationID,fare_amount,Surge Fee,Vehicle,Payment_type
0,1,NaT,NaT,1,5.60,79,226,19.5,2.0,UberX,Uber Pay
1,2,NaT,NaT,1,1.72,142,186,8.0,0.0,Uber Black,Cash
2,3,NaT,NaT,1,3.41,229,238,13.0,0.0,Uber Black,Cash
3,4,NaT,NaT,1,1.81,188,35,9.0,0.0,UberX,Cash
4,5,NaT,NaT,1,1.89,100,137,8.0,0.0,Uber Black,Cash


In [17]:
# Save the current working copies so they can be reused in later notebooks
trips_clean.to_csv("Uber_Trip_Details_Validated.csv", index=False)
locations_clean.to_csv("Location_Table_Validated.csv", index=False)

# Confirm that the output files were created
print("Validated datasets saved successfully.")

Validated datasets saved successfully.
